# 11b · BGE Reranker v2 M3 (Cross-Encoder)

**Tujuan:** Mengevaluasi pendekatan Cross-Encoder Reranker (`BAAI/bge-reranker-base` / `bge-reranker-v2-m3`).
Berbeda dengan Bi-Encoder yang menghitung vektor embedding secara terpisah, Cross-Encoder menerima pasangan `(Judul + Abstrak, Profil Kompetensi)` secara bersamaan ke dalam transformer layer dengan full cross-attention untuk menghasilkan skor relevansi semantik yang lebih presisi.

## 0. Setup dan validasi sumber data

In [ ]:
from pathlib import Path
import os, sys, json, time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
assert (ROOT / 'data' / 'dataset_buku_2000.csv').exists(), 'Data folder not found'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA, OUT, FIG, MODELS = [ROOT / x for x in ('data', 'outputs', 'figures', 'models')]
for f in (OUT, FIG, MODELS):
    f.mkdir(exist_ok=True)

from src import engine
books, comps = engine.load_data()
print(f'Dataset: {len(books)} buku | {len(comps)} kompetensi')
assert books['book_id'].is_unique and comps['competency_id'].is_unique

## 1. Arsitektur Cross-Encoder Reranker vs Bi-Encoder

In [ ]:
from src.models.cross_encoder_model import CrossEncoderRerankerModel

reranker = CrossEncoderRerankerModel(comps, pretrained_id='BAAI/bge-reranker-base')
print('Model Pretrained ID:', reranker.pretrained_id)
print('Kompetensi:', len(reranker.competencies))
print('Mekanisme: Full cross-attention layer per (query, passage) pair.')

## 2. Pengecekan Skor Pasangan Lengkap (65.191 Pasangan)

In [ ]:
reranker_path = OUT / 'skor_semua_pasangan_bge_reranker.csv'
assert reranker_path.exists(), 'File skor pasangan bge_reranker belum tersedia'
df_reranker = pd.read_csv(reranker_path, dtype={'book_id': str, 'competency_id': str})
print(f'Total baris pasangan: {len(df_reranker):,}')
assert len(df_reranker) == len(books) * len(comps)

stats = df_reranker['skor_asli'].describe()
display(pd.DataFrame(stats).T)

## 3. Audit Ranking Buku untuk Sampel Kompetensi

In [ ]:
sample_cid = 'C023'  # Manajemen Risiko
top_c023 = (df_reranker[df_reranker['competency_id'] == sample_cid]
            .sort_values('skor_asli', ascending=False)
            .head(10)
            .merge(books[['book_id', 'judul']], on='book_id', how='left'))
top_c023['ranking'] = range(1, len(top_c023) + 1)
print(f'Top 10 Buku untuk Kompetensi {sample_cid}:')
display(top_c023[['ranking', 'book_id', 'judul', 'skor_asli']])

## Kesimpulan Tahap

1. Skor pasangan Cross-Encoder telah dihitung secara lengkap (65.191 pasangan).
2. Cross-Encoder memberikan pemisahan relevansi semantik yang sangat tajam.
3. Trade-off komputasi: Memerlukan full transformer pass untuk setiap pasangan buku-kompetensi, sehingga sangat cocok sebagai reranker tahap akhir atau pembanding benchmark kualitas tinggi.